# Example - Multiparticle tracking optimization

In this notebook, we'll set up an optimization problem. The objective will be to set the BC20 sextupole strengths (symmetrically) to minimize the spot size at PENT

For this demo, we'll start from a reference configuration. To speed up the notebook, CSR is disabled

checkpointElement is used to define the trackStart; since we're only changing elements in BC20, we'll start tracking there instead of tracking the whole lattice

This example is wraps the optimization in Example - Multiparticle tracking optimization in Xopt.  Based on xopt example here: https://xopt.xopt.org/examples/basic/xopt_basic/

In [1]:
from xopt import Evaluator
from xopt import VOCS
from xopt import Xopt
from xopt.generators import list_available_generators
from xopt.generators import get_generator
import math
from pmd_beamphysics import ParticleGroup

import UTILITY_quickstart as qs

import numpy as np
import pandas as pd




/sdf/home/c/cropp/FACET2-S2E/UTILITY_plotMod.py:347: SyntaxWarning: invalid escape sequence '\s'
  sep="[\s\n]+",


In [2]:
importedDefaultSettings = qs.loadConfig("setLattice_configs/2024-10-22_oneBunch_baseline3.yml")

In [3]:
csrTF = False
evalElement = "PENT"

inputBeamFilePathSuffix = importedDefaultSettings["inputBeamFilePathSuffix"]
bunchCount = importedDefaultSettings["bunchCount"]

tao = qs.initializeTao(
    inputBeamFilePathSuffix = inputBeamFilePathSuffix,
    
    csrTF = csrTF,
    numMacroParticles=1e3,
    scratchPath = "/tmp",
    randomizeFileNames = True
)


#Set aside the initial beam for later reference
qs.trackBeam(tao, trackEnd = "L0BFEND")
PInit = ParticleGroup(data=tao.bunch_data("L0AFEND"))

#Set lattice to imported config
qs.setLattice(tao, **importedDefaultSettings) 

Environment set to:  /sdf/home/c/cropp/FACET2-S2E
CSR off
Overwriting lattice with setLattice() defaults
No defaults file provided to setLattice(). Using /sdf/home/c/cropp/FACET2-S2E/setLattice_configs/defaults.yml
Number of macro particles = 1000.0


OSError: Unable to synchronously open file (file signature not found)

In [ ]:
# targetBunchSpacing is for two-bunch optimization
# targetBunchSpacing = 40e-6



# For some optimizations, it's useful for the tolerances to evolve over time
masterToleranceScalingStart = 1 #Higher is looser; generally tighten for early sims and loosen for refinement
masterToleranceScalingEnd = 1 #masterToleranceScalingStart
masterToleranceScalingEvolutionSteps = 2000
masterToleranceScaling = masterToleranceScalingStart



# Enable or disable different penalty terms
enableAlignmentTerms    = 1                        
    # Enable centroid offset and angle penalties. Useful for removing final focus kickers from free parameters
enableLongitudinalTerms = 1                       
    # Enable bunch spacing penalty and bunch length objectives 
alignmentIsRelative     = 1 and (bunchCount == 2)  
    # Toggle between absolute and relative pointing. Must be absolute for single bunch or stuff breaks
stageOneOptimization    = 0                        
    # Disabling transverse terms and length limits, per https://docs.google.com/presentation/d/1b6WoEwmDz5cA0fm9FbbGiZoMwBcbCNNNCsD7JiFDxRc/edit#slide=id.g2f91233284b_0_0



# If usingCheckpoint, partial tracking is used from checkpointElement to evalElement
# Select the appropriate element based on the active free parameters
usingCheckpoint = True
#checkpointElement = "MFFF"
checkpointElement = "CB1LE" #Shortly downstream of BEGBC20; there's an intervening dipole though
#checkpointElement = "BEGBC20"
#checkpointElement = "HLAM"  #Upstream of Q19851


In [ ]:
## Define VOCS

pbounds = {
    # "QA10361kG": eval(importedDefaultSettings["QA10361kGBounds"]),
    # "QA10371kG": eval(importedDefaultSettings["QA10371kGBounds"]),
    # "QE10425kG": eval(importedDefaultSettings["QE10425kGBounds"]),
    # "QE10441kG": eval(importedDefaultSettings["QE10441kGBounds"]),
    # "QE10511kG": eval(importedDefaultSettings["QE10511kGBounds"]),
    # "QE10525kG": eval(importedDefaultSettings["QE10525kGBounds"]),

    # # 'L0BPhaseSet': (-30, 30),
    # 'L1PhaseSet': (-60, 0),
    # 'L2PhaseSet': (-60, 0),
    # 'L3PhaseSet': (-40, 40),
    
    # # # 'L0BEnergyOffset': (-5e6, 5e6),
    # 'L1EnergyOffset': (-20e6, 20e6),
    # 'L2EnergyOffset': (-500e6, 500e6),
    # 'L3EnergyOffset': (-4500e6, -3500e6),

    # "Q19851kG":  eval(importedDefaultSettings["Q19851kGBounds"]),
    # "Q19871kG":  eval(importedDefaultSettings["Q19871kGBounds"]),

    
    # "B1EkG" : (4.4, 4.6),      #(4.0,  5.0),        
    # "B2EkG" : (-6.65, -6.45),  #(-7.0, -6.0),       
    # "B3EkG" : (1.95, 2.15),    #(1.5,  2.5),        
    
    # "Q1EkG":  eval(importedDefaultSettings["Q1EkGBounds"]),
    # "Q2EkG":  eval(importedDefaultSettings["Q2EkGBounds"]),
    # "Q3EkG":  eval(importedDefaultSettings["Q3EkGBounds"]),
    # "Q4EkG":  eval(importedDefaultSettings["Q4EkGBounds"]),
    # "Q5EkG":  eval(importedDefaultSettings["Q5EkGBounds"]),
    # "Q6EkG":  eval(importedDefaultSettings["Q6EkGBounds"]),
    
    "S1ELkG": eval(importedDefaultSettings["S1ELkGBounds"]),
    "S2ELkG": eval(importedDefaultSettings["S2ELkGBounds"]),
    "S3ELkG": eval(importedDefaultSettings["S3ELkGBounds"]),
    # "S3ERkG": eval(importedDefaultSettings["S3ERkGBounds"]),
    # "S2ERkG": eval(importedDefaultSettings["S2ERkGBounds"]),
    # "S1ERkG": eval(importedDefaultSettings["S1ERkGBounds"]),

    # "S1EL_xOffset" : ( -0.004, 0.004 ),  
    # "S1EL_yOffset" : ( -0.004, 0.004 ),  
    # "S2EL_xOffset" : ( -0.004, 0.004 ),  
    # "S2EL_yOffset" : ( -0.004, 0.004 ),  
    # "S2ER_xOffset" : ( -0.004, 0.004 ),  
    # "S2ER_yOffset" : ( -0.004, 0.004 ),  
    # "S1ER_xOffset" : ( -0.004, 0.004 ),  
    # "S1ER_yOffset" : ( -0.004, 0.004 ),

    # 'Q5FFkG': eval(importedDefaultSettings["Q5FFkGBounds"]),
    # 'Q4FFkG': eval(importedDefaultSettings["Q4FFkGBounds"]),
    # 'Q3FFkG': eval(importedDefaultSettings["Q3FFkGBounds"]),
    # 'Q2FFkG': eval(importedDefaultSettings["Q2FFkGBounds"]),
    # 'Q1FFkG': eval(importedDefaultSettings["Q1FFkGBounds"]),
    # 'Q0FFkG': eval(importedDefaultSettings["Q0FFkGBounds"]),
    # 'Q0DkG':  eval(importedDefaultSettings["Q0DkGBounds"]),
    # 'Q1DkG':  eval(importedDefaultSettings["Q1DkGBounds"]),
    # 'Q2DkG':  eval(importedDefaultSettings["Q2DkGBounds"]),

    # "XC1FFkG" : tuple(2 * x for x in eval(importedDefaultSettings["XC1FFkGBounds"])), #2024-10-11: Extending bounds as proxy for tuning final chicane dipole strength
    # "XC3FFkG" : eval(importedDefaultSettings["XC3FFkGBounds"]),
    # "YC1FFkG" : eval(importedDefaultSettings["YC1FFkGBounds"]),
    # "YC2FFkG" : eval(importedDefaultSettings["YC2FFkGBounds"]),
}




In [ ]:
## Objective

def specificOptimizer(
    self,
    **kwargs
):

    self.totalNumEvals += 1
    self.displayEvals()

    qs.updateMasterToleranceScaling(self.totalNumEvals)

    
    savedData = kwargs
    
    badValue = -1e30  #The value returned for illegal config. Should be colossal. Double limit ~= 1e308
    bigCost  = 1e20   #Should be large enough to dominate any "normal" return value but be dominated by badValue
    
    try: #This try block deals with bad configurations. Instead of causing the optimizer to halt we now 'except' a low value
        qs.setLattice(tao, **( importedDefaultSettings |  kwargs ))

    except:
        print(f"specificOptimizer() excepted'd on setLattice()")
        return badValue * 5

    try:
        if usingCheckpoint: 
            qs.trackBeam(tao, 
                      trackStart = checkpointElement, 
                      trackEnd = evalElement, 
                      **importedDefaultSettings)
        else:
            qs.trackBeam(tao, 
                      trackStart = "L0AFEND", 
                      trackEnd = evalElement, 
                      **importedDefaultSettings)

    except:
            print(f"specificOptimizer() excepted'd on trackBeam()")
            return badValue * 4
    

    if tao.bunch_params(evalElement)['n_particle_live'] < 10:
        print(f"specificOptimizer() got ~no particles after tracking")
        return badValue * 2 

    
    
    try: 
        P = qs.getBeamAtElement(tao, evalElement)
        savedData = savedData | qs.getBeamSpecs(P, targetTwiss = evalElement)  
        savedData["lostChargeFraction"] = 1 - (P.charge / PInit.charge)

    except:
        print(f"specificOptimizer() excepted'd while getting beam and compiling savedData")
        return badValue



    
    try:
        if stageOneOptimization: 
            enableTransverse = 0
            lengthLimitMultiplier = 0
            stageOneSpacingToleranceMultiplier = 0.5
    
        else:
            enableTransverse = 1
            lengthLimitMultiplier = 1


        #######################################################################
        #######################################################################
        # User specified objective
        #######################################################################
        #######################################################################


        # Tolerances. Generally apply high penalty if tolerances are not satisfied but no cost if they are
        tolerableBeamLossFraction  = 0.02  * masterToleranceScaling
        tolerableBunchSpacingError = 10e-6 * masterToleranceScaling * (stageOneSpacingToleranceMultiplier if stageOneOptimization else 1.0)
        
        tolerableBeamOffset  = 20e-6 * masterToleranceScaling #5e-6
        tolerableAngleOffset = 20e-3 * masterToleranceScaling #5e-3
        
        driveEmittanceThreshold   = 15e-6 * masterToleranceScaling #15 um-rad is 20 um at 50 cm beta
        witnessEmittanceThreshold = 15e-6 * masterToleranceScaling
    
        driveSpotThreshold     = 20e-6 #* masterToleranceScaling
        witnessSpotThreshold   = 20e-6 #* masterToleranceScaling
        
        driveLengthThreshold   = lengthLimitMultiplier * 20e-6 * masterToleranceScaling
        witnessLengthThreshold = lengthLimitMultiplier * 20e-6 * masterToleranceScaling
    
        slicewiseBMAGThreshold = 1 + ( 0.2 * masterToleranceScaling ) #1.1
        

        # Penalize losing charge
        savedData["errorTerm_lostChargeFraction"] = 1e3 * qs.rampToZero( savedData["lostChargeFraction"], tolerableBeamLossFraction, scale = 0.01)**2

        # Penalize incorrect two bunch spacing
        savedData["errorTerm_bunchSpacing"] = (
            enableLongitudinalTerms * 1e3 * qs.rampToZero( abs(savedData["bunchSpacing"] - qs.targetBunchSpacing), tolerableBunchSpacingError, scale = 10e-6)**2
            if bunchCount == 2
            else
            0
        )


        # Penalize bad alignment
        if alignmentIsRelative:
            
            savedData["errorTerm_transverseOffset"] = enableAlignmentTerms * enableTransverse * 1e3 * np.mean([
                        qs.rampToZero(abs(savedData["PDrive_median_x"]  - savedData["PWitness_median_x"] ), tolerableBeamOffset, scale = 1e-6) ** 2,
                        qs.rampToZero(abs(savedData["PDrive_median_y"]  - savedData["PWitness_median_y"] ), tolerableBeamOffset, scale = 1e-6) ** 2,
            ])
            
            savedData["errorTerm_angleOffset"] = enableAlignmentTerms * enableTransverse * 1e3 * np.mean([
                        qs.rampToZero(abs(savedData["PDrive_median_xp"]  - savedData["PWitness_median_xp"] ), tolerableAngleOffset, scale = 100e-6) ** 2,
                        qs.rampToZero(abs(savedData["PDrive_median_yp"]  - savedData["PWitness_median_yp"] ), tolerableAngleOffset, scale = 100e-6) ** 2,
            ])
        
        else:
        
            savedData["errorTerm_transverseOffset"] = enableAlignmentTerms * enableTransverse * 1e3 * np.mean([
                        (qs.rampToZero(abs(savedData["PWitness_median_x"]), tolerableBeamOffset, scale = 1e-6) ** 2 if bunchCount == 2 else 0),
                        (qs.rampToZero(abs(savedData["PWitness_median_y"]), tolerableBeamOffset, scale = 1e-6) ** 2 if bunchCount == 2 else 0),
                        qs.rampToZero(abs(savedData["PDrive_median_x"]  ), tolerableBeamOffset, scale = 1e-6) ** 2,
                        qs.rampToZero(abs(savedData["PDrive_median_y"]  ), tolerableBeamOffset, scale = 1e-6) ** 2,
            ])
            
            savedData["errorTerm_angleOffset"] = enableAlignmentTerms * enableTransverse * 1e3 * np.mean([
                        (qs.rampToZero(abs(savedData["PWitness_median_xp"]), tolerableAngleOffset, scale = 100e-6) ** 2 if bunchCount == 2 else 0),
                        (qs.rampToZero(abs(savedData["PWitness_median_yp"]), tolerableAngleOffset, scale = 100e-6) ** 2 if bunchCount == 2 else 0),
                        qs.rampToZero(abs(savedData["PDrive_median_xp"]  ), tolerableAngleOffset, scale = 100e-6) ** 2,
                        qs.rampToZero(abs(savedData["PDrive_median_yp"]  ), tolerableAngleOffset, scale = 100e-6) ** 2,
            ])
        
    
        #2024-11-25-15-58-15: Trying to prevent optimizer from "rolling up" phase space at PENT
        #savedData["errorTerm_sigma_xp_rule"] = 1e3 * ( rampToZeroFlip(savedData[f"PDrive_sigmaSI90_xp"], 500e-6, 10e-6) ) ** 2
    
        #2024-11-26-12-38-20: Force the beam to match design twiss
        #savedData["errorTerm_BMAG_rule"] = 1e3 * ( rampToZero(savedData[f"PDrive_BMAG_x"], 1.1, 1) + rampToZero(savedData[f"PDrive_BMAG_y"], 1.1, 1)) ** 2
       
        #2024-11-27-10-11-23: Require each slice to comply
        # savedData["errorTerm_BMAG_rule"] = enableTransverse * 1e3 * (
        #     np.sum([rampToZero(activeBMAG, slicewiseBMAGThreshold, 1) ** 2 for activeBMAG in savedData[f"PDrive_sliced_BMAG_x"]]) + 
        #     np.sum([rampToZero(activeBMAG, slicewiseBMAGThreshold, 1) ** 2 for activeBMAG in savedData[f"PDrive_sliced_BMAG_y"]])
        # )

        #2025-04-04-17-05-10: Special term to prevent optimizer from slipping into pencil beam
        #savedData["errorTerm_minimalEnergySpread"] = 1e3 * ( rampToZeroFlip( P["sigma_energy"],  90e6 ) ) ** 2
    

        # The mainObjective defines the primary goal of the optimization, subject to the constraints imposed with the penalties above
        # Uncomment or add terms as required
        savedData["errorTerm_mainObjective"] = np.mean([

                     enableLongitudinalTerms *                   qs.rampToZero(savedData[f"PDrive_sigmaSI90_z"],   driveLengthThreshold,   10e-6) ** 2,
                    (enableLongitudinalTerms *                   qs.rampToZero(savedData[f"PWitness_sigmaSI90_z"], witnessLengthThreshold, 10e-6) ** 2 if bunchCount == 2 else 0),
    
                    #  enableTransverse *                           rampToZero(savedData[f"PDrive_sigmaSI90_x"],   driveSpotThreshold,     10e-6) ** 2,
                    #  enableTransverse *                           rampToZero(savedData[f"PDrive_sigmaSI90_y"],   driveSpotThreshold,     10e-6) ** 2,
                    # (enableTransverse *                           rampToZero(savedData[f"PWitness_sigmaSI90_x"], witnessSpotThreshold,   10e-6) ** 2 if bunchCount == 2 else 0),
                    # (enableTransverse *                           rampToZero(savedData[f"PWitness_sigmaSI90_y"], witnessSpotThreshold,   10e-6) ** 2 if bunchCount == 2 else 0),
    
                    # enableTransverse *                           rampToZero(savedData[f"PDrive_emitSI90_x"],   driveEmittanceThreshold,     10e-6) ** 2,
                    # enableTransverse *                           rampToZero(savedData[f"PDrive_emitSI90_y"],   driveEmittanceThreshold,     10e-6) ** 2,
                    # (enableTransverse *                          rampToZero(savedData[f"PWitness_emitSI90_x"], witnessEmittanceThreshold,   10e-6) ** 2 if bunchCount == 2 else 0),
                    # (enableTransverse *                          rampToZero(savedData[f"PWitness_emitSI90_y"], witnessEmittanceThreshold,   10e-6) ** 2 if bunchCount == 2 else 0),
    
                    # rampToZero(savedData[f"PDrive_emitSI90_x"], 0, 10e-6) ** 2,
                    # rampToZero(savedData[f"PDrive_emitSI90_y"], 0, 10e-6) ** 2,
    
                    # rampToZero(savedData[f"PDrive_norm_emit_x"], 0, 10e-6) ** 2,
                    # rampToZero(savedData[f"PDrive_norm_emit_y"], 0, 10e-6) ** 2,
    
                    #enableTransverse * rampToZero(savedData[f"PDrive_norm_emit_x"] * savedData[f"PDrive_BMAG_x"], driveEmittanceThreshold, 10e-6) ** 2,
                    #enableTransverse * rampToZero(savedData[f"PDrive_norm_emit_y"] * savedData[f"PDrive_BMAG_y"], driveEmittanceThreshold, 10e-6) ** 2,
        ]) 
    
        # Secondary objective includes all "ramp" terms with thresholds disabled. Intended to gently nudge all specs to better values if nothing else is going on; mostly expect this to do anything once thresholds are hit
        # Uncomment or add terms as required
        # 2024-10-15 comment: Advise not setting this weight above 1e-6. For quite-good mainObjective settings, even 1e-4 is too much
        savedData["errorTerm_secondaryObjective"] = 1e-6 * np.mean([
                    #(rampToZero( abs(savedData["bunchSpacing"] - targetBunchSpacing), 0 * tolerableBunchSpacingError, scale = 10e-6) ** 2 if bunchCount == 2 else 0),
    
                    # rampToZero(abs(savedData["PDrive_median_x"]  ),                   0 * tolerableBeamOffset, scale = 1e-6) ** 2,
                    # rampToZero(abs(savedData["PDrive_median_y"]  ),                   0 * tolerableBeamOffset, scale = 1e-6) ** 2,
                    # (rampToZero(abs(savedData["PWitness_median_x"]),                  0 * tolerableBeamOffset, scale = 1e-6) ** 2 if bunchCount == 2 else 0),
                    # (rampToZero(abs(savedData["PWitness_median_y"]),                  0 * tolerableBeamOffset, scale = 1e-6) ** 2 if bunchCount == 2 else 0),
    
                    # rampToZero(abs(savedData["PDrive_median_xp"]  ),                  0 * tolerableAngleOffset, scale = 100e-6) ** 2,
                    # rampToZero(abs(savedData["PDrive_median_yp"]  ),                  0 * tolerableAngleOffset, scale = 100e-6) ** 2,
                    # (rampToZero(abs(savedData["PWitness_median_xp"]),                 0 * tolerableAngleOffset, scale = 100e-6) ** 2 if bunchCount == 2 else 0),
                    # (rampToZero(abs(savedData["PWitness_median_yp"]),                 0 * tolerableAngleOffset, scale = 100e-6) ** 2 if bunchCount == 2 else 0),
    
                    qs.rampToZero(savedData[f"PDrive_sigmaSI90_x"],                      0 * driveSpotThreshold,     10e-6) ** 2,
                    qs.rampToZero(savedData[f"PDrive_sigmaSI90_y"],                      0 * driveSpotThreshold,     10e-6) ** 2,
                    qs.rampToZero(savedData[f"PDrive_sigmaSI90_z"],                      0 * driveLengthThreshold,   10e-6) ** 2,
                    (qs.rampToZero(savedData[f"PWitness_sigmaSI90_x"],                   0 * witnessSpotThreshold,   10e-6) ** 2 if bunchCount == 2 else 0),
                    (qs.rampToZero(savedData[f"PWitness_sigmaSI90_y"],                   0 * witnessSpotThreshold,   10e-6) ** 2 if bunchCount == 2 else 0),
                    (qs.rampToZero(savedData[f"PWitness_sigmaSI90_z"],                   0 * witnessLengthThreshold, 10e-6) ** 2 if bunchCount == 2 else 0),
        ])


        # Combine all errorTerms
        savedData["maximizeMe"] = 1 / np.mean([
            #Constraints
            savedData["errorTerm_lostChargeFraction"], 
            savedData["errorTerm_bunchSpacing"],
            savedData["errorTerm_transverseOffset"],
            savedData["errorTerm_angleOffset"],
    
            #Objectives
            savedData["errorTerm_mainObjective"],
            savedData["errorTerm_secondaryObjective"],
    
            #Additional, specialized terms
            #savedData["errorTerm_sigma_xp_rule"],
            (savedData["errorTerm_BMAG_rule"] if "errorTerm_BMAG_rule" in savedData else 0),
            (savedData["errorTerm_minimalEnergySpread"] if "errorTerm_minimalEnergySpread" in savedData else 0),
            
            1e-20 #Avoid infinities 
        ])
    
        savedData["inputBeamFilePathSuffix"] = inputBeamFilePathSuffix
        savedData["csrTF"] = csrTF

    except:
        print(f"specificOptimizer() excepted'd while calculating error terms")
        return 0.9 * badValue
    

    
    #Collect desired data as a pandas Series
    tmpData = pd.Series( savedData ) 
    self.history = pd.concat([self.history, tmpData.to_frame().T])

    #Optional: Write to file
    #self.history.to_json('optimizerHistory.json', orient='records')
    n = self.totalNumEvals
    if (n < 100) or (n < 10000 and  n % 10 == 0) or (n % 100 == 0): #This gets expensive to write when n >> 10k
        self.history.to_json('optimizerHistory.json', orient='records')
    
    self.updatePlot()

    
    


    sortedHistory = self.history.sort_values(by='maximizeMe', ascending=False)
    bestConfigData = sortedHistory.iloc[0]

    #Optionally: Write beams for running best case
    if savedData["maximizeMe"] == bestConfigData["maximizeMe"]:
        
        (qs.getBeamAtElement(tao, "ENDBC14_2")).write("beams/optimizerRunningBestBeam_ENDBC14.h5")
        (qs.getBeamAtElement(tao, "BEGBC20")).write("beams/optimizerRunningBestBeam_BEGBC20.h5")
        (qs.getBeamAtElement(tao, "MFFF")).write("beams/optimizerRunningBestBeam_MFFF.h5")
        (qs.getBeamAtElement(tao, "PENT")).write("beams/optimizerRunningBestBeam_PENT.h5")

        # print("Writing PENT")
        # print(getBeamAtElement(tao, "PENT"))


    

    
    return savedData

evaluator = Evaluator(function=specificOptimizer)

In [ ]:
vocs = VOCS(
    variables = pbounds,
    objectives = {"maximizeMe": "MAXIMIZE"}
    #Constraints = {}
)

In [ ]:
list_available_generators()

In [ ]:
# get the docstring for the random generator
print(get_generator("random").__doc__)

# use the get generator method to get the random number generator
generator = get_generator("neldermead")(vocs=vocs)

In [ ]:
X = Xopt(vocs=vocs, generator=generator, evaluator=evaluator)

In [ ]:
# Can change stopping criterion here, but just 100 runs of the optimizer for now

import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

for i in range(100):
    X.step()
    clear_output(wait=True)
    plt.figure()
    for obj in vocs.objectives.keys():
        plt.plot(X.data[obj])
    plt.title(f"Iteration {i}")
    plt.show()
